# A1 & A2 · DFS events: what is the data?

Step 1: just look at the raw DFS event files NESO publishes for 2022/23. The last
section introduces the national dip test and scopes the data for both groups:
A1 (2022/23) and A2 (Oct 2023+).

The first code cell downloads any raw data that's missing (`data/` is not committed), so
this notebook runs from a fresh clone. Loaders: `uk_dfs.data.neso` (DFS events, national
demand) and `uk_dfs.data.elexon` (5-minute generation).

In [ ]:
import pandas as pd

from uk_dfs.config import RAW_DIR
from uk_dfs.data import elexon, neso

# Download any raw data that isn't on disk yet (first run: a few minutes; after that: instant)
neso.fetch_dfs()
neso.fetch_demand()
elexon.fetch_fuelinst("2022-11")  # 5-minute data from the first DFS season to today

DFS = RAW_DIR / "neso_dfs"

## Live events
One row per half-hour of each event.

In [ ]:
live = pd.read_csv(DFS / "2223_live_summary.csv")
live

## Test events
Same columns. In 2022/23 most events were "tests" run to build up participation.

In [ ]:
test = pd.read_csv(DFS / "2223_test_summary.csv")
test

## 5-minute generation (Elexon FUELINST)
One Parquet file per month in `data/raw/elexon_fuelinst/`. "Long" format: one row per
5-minute interval × fuel type, with generation in MW. Interconnectors (`INT…`) and pumped
storage (`PS`) go negative when GB is exporting or pumping.

In [ ]:
fuel = pd.read_parquet(RAW_DIR / "elexon_fuelinst" / "2023-01.parquet")
print(fuel.shape)
print(fuel.fuelType.unique())
fuel.head(20)

### One day, one row per 5 minutes
Reshape to "wide": one row per 5-minute interval, one column per fuel type, plus a total.
24 Jan 2023 is the second live event (16:30–18:00). Times are UTC, which in January is
the same as UK clock time.

In [ ]:
day = fuel[fuel.startTime.dt.date == pd.Timestamp("2023-01-24").date()]
day = day.pivot(index="startTime", columns="fuelType", values="generation")
day["TOTAL"] = day.sum(axis=1)
day

## Exploratory plots

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

# Project colours; importing _common also applies the chart style
from _common import AQUA, BLUE, INK2, ORANGE

### One table of events
Combine live and test into one table with a `type` column and a proper start time.
Tidy the column names *before* combining: the two files spell some names differently
(trailing spaces), and combining first would split `DFS Procured` into two columns.

In [ ]:
for df in (live, test):
    df.columns = df.columns.str.strip()  # trailing spaces differ between the two files
events = pd.concat([live.assign(type="Live"), test.assign(type="Test")], ignore_index=True)
events["start"] = pd.to_datetime(events.Date + " " + events.From, dayfirst=True)
events = events.sort_values("start").reset_index(drop=True)
events[["start", "type", "DFS Required", "DFS Procured", "Settled Volume"]]

### Plot 1: how big was each event?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for kind, colour in (("Test", BLUE), ("Live", ORANGE)):
    e = events[events.type == kind]
    ax.scatter(
        e.start,
        e["Settled Volume"],
        s=36,
        color=colour,
        edgecolor="white",
        lw=0.8,
        label=kind,
        zorder=3,
    )
ax.set_ylabel("Settled volume, MW")
ax.set_title("2022/23 DFS: every event half-hour")
ax.xaxis.set_major_locator(mdates.MonthLocator())  # one tick per month: "Dec 2022", ...
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper left");

### Total generation for the whole season, every 5 minutes
Sum across fuel types for Nov 2022 – Mar 2023, converted to UK clock time. This is raw: a
handful of intervals (26 in the season) are partial publications that read far too low.
`elexon.load_fuelinst_total()` masks those; we haven't yet.

In [ ]:
season = [
    f
    for f in sorted((RAW_DIR / "elexon_fuelinst").glob("*.parquet"))
    if "2022-11" <= f.stem <= "2023-03"
]
fuel_season = pd.concat(pd.read_parquet(f) for f in season)
total = fuel_season.drop_duplicates(["startTime", "fuelType"]).groupby("startTime").generation.sum()
# Elexon times are UTC; DFS times are UK clock time. They differ after 26 Mar 2023 (BST).
total.index = total.index.tz_localize("UTC").tz_convert("Europe/London").tz_localize(None)
total.describe()

### Plot 2: one live event at 5-minute resolution
24 Jan 2023, 16:30–18:00. Try changing `DAY` to another event date from the table above.

In [ ]:
DAY = pd.Timestamp("2023-01-24")
ev = events[events.start.dt.normalize() == DAY]
t0, t1 = ev.start.min(), ev.start.max() + pd.Timedelta(minutes=30)
day_total = total[DAY + pd.Timedelta(hours=12) : DAY + pd.Timedelta(hours=22)]

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axvspan(t0, t1, color=ORANGE, alpha=0.15, lw=0, label="Event")
ax.plot(day_total.index, day_total / 1000, color=BLUE, lw=1.5, label="Total generation (5-min)")
ax.set_ylabel("GW")
ax.set_title(
    f"{DAY:%a %d %b %Y}: {ev['Settled Volume'].mean():.0f} MW settled on average " "over the event"
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.legend(loc="upper left");

### Plot 3: the event day against two guesses of "normal"
What would demand have been without the event? Two guesses, side by side:

- **Left, naive:** the same weekday in earlier weeks. We skip days that had a DFS event of
  their own (the event would pull that day's demand down) and the Christmas period; the
  next cell prints which days were used and why the others were skipped. For 24 Jan that
  pushes most comparison days back 6–8 weeks, to darker, differently-heated evenings.
- **Right, official:** the BL01 baseline used to settle DFS, applied to national
  generation as if GB were one big meter. The rules are quoted below.

Compare how far apart these guesses are with the ~0.27 GW being claimed.

In [ ]:
event_days = set(events.start.dt.normalize())
xmas = (pd.Timestamp("2022-12-19"), pd.Timestamp("2023-01-03"))

# Walk back week by week from DAY, keeping the first 4 ordinary days and saying why others go
comparison = []
for k in range(1, 13):  # up to 12 weeks back
    d = DAY - pd.Timedelta(weeks=k)
    if d in event_days:
        print(f"{d:%a %d %b}: skipped, DFS event that day")
    elif xmas[0] <= d <= xmas[1]:
        print(f"{d:%a %d %b}: skipped, Christmas period")
    else:
        print(f"{d:%a %d %b}: used")
        comparison.append(d)
    if len(comparison) == 4:
        break

### The official baseline: BL01 (Elexon P376), as used for DFS in 2022/23
Excerpt from the DFS Procurement Rules in force for the 2022/23 season, *Schedule 3 – DFS
Operational Baselines* (quoted verbatim, lightly trimmed):

> 1. Identify the required number of previous Settlement Days of the same type (Working
>    Day or Non-Working Day) for which Metered data is available. If insufficient
>    Settlement Days are available, the Baseline Value will default to the Metered data for
>    the same Settlement Period (i.e. the Baseline will equal the out-turn Metered data) […]
> 2. For each Settlement Period, calculate an Unadjusted Baseline Value (for the Unit Meter
>    Point) by averaging the Metered data values of the corresponding Settlement Period in
>    some or all of the previous Settlement Days identified in step 1.
> 3. Calculate an In Day Adjustment, and add it to the Unadjusted Baseline Value (this
>    adjustment only applies to 'Domestic' DFS Unit Type)
>
> […] identifying all "Eligible Days" for each relevant Unit Meter Point i.e. Settlement Days
> in the sixty-day window that:
> - are of the same time type (Working Day or Non-Working Day) as day D;
> - have Half-Hourly (HH) Metered data available for the Unit Meter Point
> - are not Event Days; and
> - are not 'clock change' days […]
>
> | Day Type | No. of Eligible Days identified in the 60-day window | Historical Settlement Days used to calculate the baseline |
> |---|---|---|
> | Working Day | Ten or more Eligible Days | Ten most recent Eligible Days |
> | Working Day | 5 to 9 Eligible Days | All Eligible Days |
> | Working Day | Less than five Eligible Days | Baseline Values default to out-turn metered data […] |
> | Non-Working Day | Four or more Eligible Days | Four most recent Eligible Days |
>
> […] For a Non-Working Day, two of the four Eligible Days identified in step 1 will be
> used. The two selected will be the middle 2 of the four (ranking them in order of the
> total Unit Meter Point Metered Volume, summed over the Settlement Day)
>
> The In Day Adjustment will consider actual Metered data over the three hour period up
> until one hour ahead of the relevant Settlement Period when the service starts to
> deliver. […]
>
> In Day Adjustment = ∑j (Unit Meter Point Metered Volume – Unadjusted Baseline Value) / 6
>
> where ∑j represents summation over the six Settlement Periods in the "In Day Reference
> Window". The In Day Reference Window is defined as the three-hour period ending at one
> hour ahead of the first Settlement Period in Settlement Day D.
>
> Baseline Value = Unadjusted Baseline Value + In Day Adjustment

**Source:** National Grid ESO, *Demand Flexibility Service Procurement Rules*, v1 (PDF dated
1 Sep 2022), Schedule 3. https://www.neso.energy/document/266556/download (local copy:
`data/raw/neso_docs/266556.pdf`). The method comes from Elexon's BSC modification
[P376](https://www.elexon.co.uk/bsc/documents/groups/panel/2021-meeting/312-march/312-04-p376-utilising-a-baselining-methodology-to-set-physical-notifications-for-settlement-of-applicable-balancing-services/).
The In Day Adjustment was dropped for events from 27 Oct 2023 (Procurement Rules v2).

**How we apply it here.** BL01 is meant for one household meter; we run the same rules on
national generation, treating GB as one meter. That shows what the rule does on a cold day
but is not how any provider's delivery was measured. Every day with a DFS event counts as
an Event Day. The code is `uk_dfs.models.bl01`.

In [ ]:
import numpy as np

from uk_dfs.models import bl01

# BL01 works on half-hours, so average the 5-minute series into settlement periods. Use the
# cleaned loader: one partial publication (a near-zero reading) would distort a baseline day.
hh = elexon.load_fuelinst_total().resample("30min").mean()

# One "meter" (GB) over days D-60 … D, 48 half-hours each: shape (1, 61, 48)
days = pd.date_range(DAY - pd.Timedelta(days=60), DAY)
X = np.stack([hh.reindex(pd.date_range(d, periods=48, freq="30min")).to_numpy() for d in days])
X = X[None]
is_event = days.isin(event_days)

B, n = bl01.unadjusted(X, days, event_days=is_event)
start_slot = t0.hour * 2 + t0.minute // 30  # first event half-hour, 0-based
adj = bl01.in_day_adjustment(X, B, start_slot)[0, -1]
day_slots = pd.date_range(DAY, periods=48, freq="30min")
baseline_unadj = pd.Series(B[0, -1], index=day_slots)
baseline = baseline_unadj + adj
ref0, ref1 = day_slots[start_slot - 8], day_slots[start_slot - 2]  # in-day reference window

# Which days went into the average (same rules, spelled out for printing)
complete = ~np.isnan(X[0]).any(axis=1)
eligible = bl01.working_days(days) & complete & ~is_event
used = days[:-1][eligible[:-1]][::-1][:10]
assert len(used) == n[0, -1]
print(f"Working days averaged ({len(used)}): " + ", ".join(f"{d:%d %b}" for d in used))
print(f"In-day adjustment: {adj:+,.0f} MW (actual minus baseline, {ref0:%H:%M}–{ref1:%H:%M})")

actual_hh = hh[day_slots]
in_event = (day_slots >= t0) & (day_slots < t1)
for name, b in (("unadjusted", baseline_unadj), ("with in-day adjustment", baseline)):
    print(f"Implied national 'delivery', BL01 {name}: {(b - actual_hh)[in_event].mean():+,.0f} MW")
print(f"Claimed (settled) delivery: {ev['Settled Volume'].mean():,.0f} MW")

In [ ]:
window = slice(DAY + pd.Timedelta(hours=12), DAY + pd.Timedelta(hours=22))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

# Left: the naive same-weekday comparison
for d in comparison:
    x = total[d + pd.Timedelta(hours=12) : d + pd.Timedelta(hours=22)]
    ax1.plot(x.index - d + DAY, x / 1000, color=INK2, lw=1, alpha=0.6)  # shift onto DAY's axis
ax1.plot(
    [],
    [],
    color=INK2,
    lw=1,
    alpha=0.6,
    label="Same weekday, earlier: " + ", ".join(f"{d:%d %b}" for d in comparison),
)
ax1.set_title("Naive: same weekday in previous weeks")

# Right: BL01, half-hourly steps
ax2.axvspan(ref0, ref1, color=AQUA, alpha=0.12, lw=0, label="In-day reference window")
ax2.step(
    baseline_unadj[window].index,
    baseline_unadj[window] / 1000,
    where="post",
    color=INK2,
    lw=1.2,
    ls="--",
    label="BL01, unadjusted",
)
ax2.step(
    baseline[window].index,
    baseline[window] / 1000,
    where="post",
    color=AQUA,
    lw=1.8,
    label=f"BL01 + in-day adjustment ({adj / 1000:+.1f} GW)",
)
ax2.set_title("Official: BL01 baseline (GB as one meter)")

for ax in (ax1, ax2):
    ax.axvspan(t0, t1, color=ORANGE, alpha=0.15, lw=0, label="Event")
    ax.plot(day_total.index, day_total / 1000, color=BLUE, lw=2, label=f"{DAY:%d %b} (event day)")
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(loc="lower center", fontsize=8)
ax1.set_ylabel("GW");

## Does an event show up in 5-minute national data? Scoping A1 and A2
The national check asks a simple question: when a DFS event starts, does GB demand fall
by roughly the volume providers claim? We test it on two groups of events with **the same
method**, split at 27 Oct 2023, when the in-day adjustment was removed from BL01:

- **A1: 2022/23**, in-day adjustment **on** for domestic meters;
- **A2: Oct 2023 onwards**, adjustment **off**.

If the two give different answers, we explain why afterwards. This section works through
one event to motivate the method, states the method precisely, then maps out how much
data each group gives us. No results yet.

In [ ]:
import numpy as np
from _common import GRID, YELLOW, load_demand, load_dfs, load_weather

summary, _ = load_dfs()  # every NESO release, one row per event half-hour

REF, GAP, MAX_LEN = 60, 10, 120  # reference hour, gap before the event, max length (min)

# 5-minute generation for every day; > 2 GW jumps flag publication glitches (see notebook 02)
F = elexon.load_fuelinst_total()
f5 = pd.DataFrame({"v": F, "bad": F.diff().abs() > 2000, "date": F.index.normalize(),
                   "m": F.index.hour * 60 + F.index.minute})  # fmt: skip
F5 = f5.pivot_table(index="date", columns="m", values="v")
BAD = f5.pivot_table(index="date", columns="m", values="bad", aggfunc="max")
BAD = BAD.fillna(True).astype(bool)

# Daily temperature and the pool of non-event days, as in notebook 02
hh = load_demand()[["date"]].join(load_weather()[["temp"]], how="left")
hh = hh[hh.date < "2026-09-13"]  # end of published demand
daily_temp = hh.groupby("date").temp.mean()
days = pd.DatetimeIndex(daily_temp.index)
non_event_days = days[~days.isin(summary[summary.proc > 0].date.unique())]


def nearby(d):
    """Non-event days within ±21 days of d with the same day type (before temperature)."""
    near = np.abs((non_event_days - d).days) <= 21
    same_type = (non_event_days.dayofweek < 5) == (d.dayofweek < 5)
    return non_event_days[near & same_type]


def matched(d):
    c = nearby(d)
    return c[np.abs(daily_temp[c].to_numpy() - daily_temp[d]) <= 2]


def clean(dates, m0, m1):
    """Days with complete, glitch-free 5-minute data from the reference hour to an hour after."""
    span = np.arange(m0 - GAP - REF, m1 + GAP + REF, 5)
    dates = dates.intersection(F5.index)
    ok = ~BAD.loc[dates, span].any(axis=1) & F5.loc[dates, span].notna().all(axis=1)
    return dates[ok.to_numpy()]


def did_panel(day, m0, m1):
    """Event day + its matched days, each minus its own reference-period mean (MW).

    Reference period: the REF minutes ending GAP minutes before the event starts. Columns
    run from there to an hour after the event (minutes after midnight).
    """
    ref = np.arange(m0 - GAP - REF, m0 - GAP, 5)
    Y = F5.loc[clean(matched(day).union([day]), m0, m1), np.arange(ref[0], m1 + GAP + REF, 5)]
    return Y.sub(Y[ref].mean(axis=1), axis=0)


def did_estimate(Y, day, m0, m1):
    """δ, MW (+ = reduction): the event day's change into the window minus the matched days'."""
    change = Y[np.arange(m0, m1, 5)].mean(axis=1)
    return -(change[day] - change.drop(day).mean())

### Plot 4: one event up close
Take one event: **Wed 30 Nov 2022, 17:30–18:30**, 217 MW claimed. Compare it with its
**matched days**: days with no DFS event within ±21 days, also weekdays, with daily mean
temperature within 2 °C.

How it was picked: a typical A1 event (one hour, at the usual time, a large claim) whose
estimate sits mid-range among A1 events. That its estimate lands almost exactly on its
claim is luck: single A1 events range from about −800 to +1,700 MW, so one event on its
own says little, and the test pools them.

**Left:** raw 5-minute generation. The matched days sit up to ~2 GW away from the event day,
many times the claimed 0.2 GW, so comparing levels directly is hopeless.

**Right:** so measure each day against **its own hour before the event** (grey), skipping
the 10 minutes just before, when early responders start. This removes each day's level.
The matched days show how demand normally moves from that hour into the event window at
this time of day (here they barely move). The estimate δ is how much further the event
day fell.

In [ ]:
EX = pd.Timestamp("2022-11-30")
ex = events[events.start.dt.normalize() == EX]  # its half-hours, from the 2022/23 table
s0, s1 = ex.start.min(), ex.start.max() + pd.Timedelta(minutes=30)
m0, m1 = s0.hour * 60 + s0.minute, s1.hour * 60 + s1.minute
Y = did_panel(EX, m0, m1)
controls = Y.index.drop(EX)
delta = did_estimate(Y, EX, m0, m1)
claim = ex["Settled Volume"].mean()
inside = np.arange(m0, m1, 5)
print(f"{len(controls)} matched days: " + ", ".join(f"{d:%d %b}" for d in controls))
print(
    f"Change from reference hour into the window: event day {Y.loc[EX, inside].mean():+.0f} MW, "
    f"matched days {Y.loc[controls, inside].mean(axis=1).mean():+.0f} MW"
)
print(f"Estimate δ: {delta:+.0f} MW reduction against {claim:.0f} MW claimed")

show = Y.columns[Y.columns < m1 + 20]  # plot up to 20 minutes after the event
raw, Y = F5.loc[Y.index, show], Y[show]
clock = EX + pd.to_timedelta(show, unit="min")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
for ax in (ax1, ax2):
    ax.axvspan(EX + pd.Timedelta(minutes=m0 - GAP - REF), EX + pd.Timedelta(minutes=m0 - GAP),
               color=GRID, alpha=0.6, lw=0)  # fmt: skip
    ax.axvspan(s0, s1, color=ORANGE, alpha=0.15, lw=0)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

for d in controls:
    ax1.plot(clock, raw.loc[d] / 1000, color=INK2, lw=0.8, alpha=0.5)
ax1.plot(clock, raw.loc[EX] / 1000, color=BLUE, lw=2)
ax1.set_ylabel("Total generation, GW")
ax1.set_title("Raw: event day vs matched days", fontsize=10)
ax1.legend(handles=[
    plt.Line2D([], [], color=BLUE, lw=2, label=f"{EX:%d %b} (event day)"),
    plt.Line2D([], [], color=INK2, lw=0.8, alpha=0.5, label=f"Matched days ({len(controls)})"),
], loc="lower center", fontsize=8)  # fmt: skip

for d in controls:
    ax2.plot(clock, Y.loc[d], color=INK2, lw=0.8, alpha=0.35)
ax2.plot(clock, Y.loc[controls].mean(), color=INK2, lw=1.8, label="Matched days, mean")
ax2.plot(clock, Y.loc[EX], color=BLUE, lw=2, label=f"{EX:%d %b} (event day)")
ax2.hlines([Y.loc[controls, inside].mean(axis=1).mean(), Y.loc[EX, inside].mean()], s0, s1,
           colors=[INK2, BLUE], lw=1.2, ls="--")  # fmt: skip
ax2.axhline(0, color=INK2, lw=0.8)
ax2.set_ylabel("Change from own reference hour, MW")
ax2.set_title(f"Each day relative to its hour before: δ = {delta:+.0f} MW vs {claim:.0f} MW "
              "claimed", fontsize=10)  # fmt: skip
ax2.legend(loc="lower left", fontsize=8)
fig.suptitle(
    "Grey: reference hour (ends 10 minutes before the event); shaded: event; "
    "dashed: window means",
    x=0.01,
    ha="left",
    fontsize=9,
    color=INK2,
);

### The method: difference-in-differences
For one event, take the event day and its matched days, and every 5-minute slot $t$ in the
**reference period** $R$ (60 minutes ending 10 minutes before the event) and the **event
window** $W$. Fit

$$y_{dt} = \alpha_d + \gamma_t - \delta \cdot E_d \cdot W_t + \varepsilon_{dt}$$

- $\alpha_d$: one level per day, which absorbs the GW-sized gaps between days;
- $\gamma_t$: one value per 5-minute slot, shared by all days: the normal shape of
  demand through this part of the evening;
- $E_d \cdot W_t$: 1 for event-day slots inside the window, 0 otherwise;
- $\delta$: the average reduction in the window, in MW (positive = demand fell).

With this layout the regression gives exactly the 2×2 **difference-in-differences** of
means that Plot 4 shows:

$$\hat\delta = -\Big[\big(\bar y_{E,W} - \bar y_{E,R}\big) - \big(\bar y_{C,W} - \bar y_{C,R}\big)\Big]$$

the event day's change from its reference hour into the window, minus the matched days'
average change.

**The assumption (parallel trends):** without the event, the event day would have moved
from its reference hour into the window the same way the matched days did on average.

**Across events.** Each event gets its own $\alpha$ and $\gamma$ (a *stacked* DiD), giving
one $\delta_e$ per event. Pooled: detected ÷ claimed $= \sum_e \delta_e / \sum_e
\text{claim}_e$. Uncertainty comes from a **bootstrap over events**, which keeps each
event's 5-minute points together (the same as clustering standard errors by event).

**Event-study chart.** Replacing $W_t$ with one indicator per 5-minute slot gives a
$\delta_k$ for every slot before, during and after the event. That's the main chart: it
shows whether the event day tracked the matched days beforehand, when the response
starts, and any rebound afterwards. The single $\delta$ is the average of the in-window
$\delta_k$.

**Placebo tests:** run the identical calculation on (a) each matched day treated as a fake
event, and (b) the event day earlier in the afternoon, where nothing should happen. These
give the spread to expect by chance.

**Two things to keep in mind:**
- *The 5-minute slots are not independent.* Deviations from the matched days are slow
  wobbles (lag-1 autocorrelation ≈ 0.9), so an hour-long window holds roughly one
  independent observation. On ordinary days, about a third of hour-long windows have
  every slot below the matched days. The evidence comes from the number of events, not
  the number of slots, hence clustering by event.
- *Noise grows with distance from the reference hour*, because event and matched days
  drift apart over the evening. Using the hour after the event as a second reference
  would cancel a straight-line drift, but it would also pick up the rebound and anything
  else that happens after events. It's reported as a robustness check, not the main
  estimate.

### What differs between the groups: the in-day adjustment and notice
Why split at Oct 2023? BL01's rules are quoted under Plot 3 above; the step that changed
is the last one:

| | A1: 2022/23 | A2: Oct 2023 onwards |
|---|---|---|
| In-day adjustment | **On** for domestic meters (off for I&C) | **Off** for all meters |
| Rule | Procurement Rules v1, Schedule 3, p. 25 | Procurement Rules v2 (effective 27 Oct 2023), p. 29; unchanged in v3 and v5 |
| Notice to providers | Day ahead: requirement ~14:30, results ~16:30 on D−1 | not checked yet |

**How the adjustment worked** (v1, p. 25): *"In Day Adjustment = ∑j (Unit Meter Point
Metered Volume – Unadjusted Baseline Value) / 6 where ∑j represents summation over the six
Settlement Periods in the 'In Day Reference Window'. The In Day Reference Window is
defined as the three-hour period ending at one hour ahead of the first Settlement Period"*
of the event. For a 17:00 event that is 13:00–16:00 on the day itself.

**Notice in 2022/23** (Winter 2022/23 review, p. 7): *"at or around 14:30 hours ESO issued
a DFS requirement […] Eligible participants submitted bids ready to be assessed at or
around 15:30 hours with notifications of results one hour later."* So providers knew
about a day ahead, and the reference window fell **after** households could have been
told. Anything they moved into the afternoon (laundry, pre-heating) or added on purpose
raised their baseline.

**Why it was removed** (NESO, Sept 2024 consultation submission, p. 11): *"The within-day
adjustment, however, can lead to perverse incentives if event notification occurs well in
advance of delivery. […] This was evidenced in the first iteration of the service."* NESO
also notes the trade-off: the adjustment *"would increase baseline accuracy if event days
conditions differed markedly from those of the recent past."*

v2 (p. 29): BL01 *"will be applied to all DFS Units without the final in day adjustment."*

**Sources** (local copies in `data/raw/neso_docs/`):
- National Grid ESO, *DFS Procurement Rules* v1 (2022/23): https://www.neso.energy/document/266556/download
- National Grid ESO, *DFS Procurement Rules* v2, effective 27 Oct 2023: https://www.neso.energy/document/286976/download
- National Grid ESO, *Demand Flexibility Service Winter 2022/23 review*, Aug 2023: https://www.neso.energy/document/287006/download
- National Grid ESO, *DFS EBR Article 18 consultation submission*, Sept 2024: https://www.neso.energy/document/324081/download

### Every turn-down event, 2022 onwards
All NESO releases, one row per event (consecutive half-hours on the same day form one
event). Tests and live events are both included: in 2022/23 the tests were paid, real
deliveries. Turn-up events (from April 2026) are out of scope. `claim` is the mean
settled MW over the event's half-hours.

In [ ]:
ADJ_OFF = pd.Timestamp("2023-10-27")  # in-day adjustment removed (Procurement Rules v2)
GROUPS = {"A1: 2022/23": BLUE, "A2: Oct 2023+": ORANGE}

print(summary[summary.proc > 0].groupby("direction").size().rename("event half-hours"))

dn = summary[(summary.direction == "Downwards") & (summary.proc > 0)].sort_values("start")
dn = dn.assign(event=(dn.start.diff() != pd.Timedelta("30min")).cumsum())
dfs_events = dn.groupby("event").agg(
    start=("start", "first"),
    halfhours=("start", "size"),
    claim=("settled", "mean"),
    has_settled=("settled", lambda s: s.notna().all()),
)
dfs_events["date"] = dfs_events.start.dt.normalize()
dfs_events["minutes"] = 30 * dfs_events.halfhours
dfs_events["m0"] = dfs_events.start.dt.hour * 60 + dfs_events.start.dt.minute
dfs_events["group"] = np.where(dfs_events.date < ADJ_OFF, *GROUPS)
dfs_events.groupby("group").agg(
    events=("start", "size"),
    event_days=("date", "nunique"),
    first=("date", "min"),
    last=("date", "max"),
    mean_claim_mw=("claim", "mean"),
).round(0)

### What the test needs from each event
The test needs the reference hour before the event, the event itself and, for the
event-study chart, the hour after; plus at least 4 matched days. An event is used only if it passes every filter below, applied in
this order:

- **settled volume published**, so there is a claim to compare with;
- **2 hours or less**, so the window stays close to the reference hour;
- **5-minute data that day**;
- **no other event near enough** to fall inside those periods or the earlier-in-the-day placebo;
- **no data glitches** over those periods;
- **at least 4 matched days** with clean data over the same minutes.

In [ ]:
STEPS = {
    "has_settled": "Settled volume published",
    "short": "Lasts 2 hours or less",
    "has_5min": "5-minute generation data that day",
    "alone": "No other event near enough to interfere",
    "clean": "No data glitches, hour before to hour after",
    "matched": "At least 4 matched days with clean data",
}
rows = []
for i, e in dfs_events.iterrows():
    m1 = e.m0 + e.minutes
    others = dfs_events[(dfs_events.date == e.date) & (dfs_events.index != i)].m0
    r = {
        "short": e.minutes <= MAX_LEN,
        "has_5min": e.date in F5.index,
        "alone": not ((others > e.m0 - 2 * MAX_LEN - 150) & (others < m1 + GAP + REF)).any(),
        "clean": False,
        "matched": False,
        "n_matched": np.nan,
    }
    fits_in_day = e.m0 - GAP - REF >= 0 and m1 + GAP + REF <= 1440
    if e.has_settled and all(r[k] for k in list(STEPS)[1:4]) and fits_in_day:
        r["clean"] = len(clean(pd.DatetimeIndex([e.date]), e.m0, m1)) == 1
        if e.date in daily_temp.index:
            r["n_matched"] = len(clean(matched(e.date), e.m0, m1))
            r["matched"] = r["n_matched"] >= 4
    rows.append(r)
checks = pd.DataFrame(rows, index=dfs_events.index)
dfs_events[checks.columns] = checks
dfs_events["used"] = dfs_events[list(STEPS)].all(axis=1)

# Funnel: events left after each filter
funnel = pd.DataFrame(
    {
        g: [len(x)] + [x[list(STEPS)[: k + 1]].all(axis=1).sum() for k in range(len(STEPS))]
        for g, x in dfs_events.groupby("group")
    },
    index=["All turn-down events"] + list(STEPS.values()),
)  # fmt: skip
funnel

### Plot 5: every event, its size, and whether the test can use it
Filled = used; hollow = dropped by one of the filters above. Log scale: claims
fell from hundreds of MW in 2022/23 to tens of MW later.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for g, colour in GROUPS.items():
    in_group = dfs_events[dfs_events.group == g]
    for used, face in ((True, colour), (False, "white")):
        x = in_group[(in_group.used == used) & (in_group.claim > 0)]
        label = f"{g}: {'used' if used else 'dropped'} ({(in_group.used == used).sum()})"
        ax.scatter(x.start, x.claim, s=28, color=face, edgecolor=colour, lw=1, label=label,
                   zorder=3)  # fmt: skip
ax.axvline(ADJ_OFF, color=INK2, ls="--", lw=1)
ax.text(ADJ_OFF, 0.03, " in-day adjustment\n removed, 27 Oct 2023", va="bottom", fontsize=8,
        color=INK2, transform=ax.get_xaxis_transform())  # fmt: skip
ax.set_yscale("log")
ax.set_ylabel("Claim: mean settled MW")
ax.set_title("Every DFS turn-down event (events with no settled volume not shown)")
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, ncol=4);

### Plot 6: how many events survive each filter
Same table as above, drawn. The groups are on different scales, so each has its own panel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, (g, colour) in zip(axes, GROUPS.items(), strict=True):
    v = funnel[g]
    y = np.arange(len(v))[::-1]
    ax.barh(y, v, color=colour, height=0.6)
    for yy, n in zip(y, v, strict=True):
        ax.text(n, yy, f" {n}", va="center", fontsize=9, color=INK2)
    ax.set_xlim(0, v.max() * 1.15)
    ax.set_title(f"{g}: {v.iloc[-1]} of {v.iloc[0]} events used", fontsize=10)
    ax.set_xlabel("Events left")
axes[0].set_yticks(y, funnel.index, fontsize=9);

### Plot 7: when the events happen
Outline = all turn-down events; filled = used in the test. Start times are UK clock time.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
for row, (g, colour) in zip(axes, GROUPS.items(), strict=True):
    x = dfs_events[dfs_events.group == g]
    panels = (
        (x.m0 / 60, np.arange(6, 22.5, 0.5), "Start time"),
        (x.minutes.clip(upper=240), np.arange(15, 255, 30), "Length, minutes (240 = 4 h or more)"),
        (x.start.dt.dayofweek, np.arange(-0.5, 7.5), "Day of week"),
    )
    for ax, (v, bins, xlabel) in zip(row, panels, strict=True):
        ax.hist(v, bins=bins, histtype="step", color=colour, lw=1.2, label="All")
        ax.hist(v[x.used], bins=bins, color=colour, alpha=0.8, label="Used in the test")
        ax.set_xlabel(xlabel, fontsize=9)
    row[0].set_ylabel(f"{g}\nevents")
    row[2].set_xticks(range(7), WEEKDAYS)
    row[1].set_xticks([30, 60, 90, 120, 180, 240])
    row[0].set_xticks(range(6, 23, 4), [f"{h:02d}:00" for h in range(6, 23, 4)])
    row[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Start time, length and day of week of every event", x=0.01, ha="left")
fig.tight_layout();

### Plot 8: the typical event, with BL01's reference window and the DiD reference hour
The most common start time and length among the events each group uses. The DiD's
reference hour overlaps only the last 10 minutes of BL01's reference window, so demand
raised in BL01's window mostly falls outside it; only extra use carried on into the hour
before the event would raise the reference and inflate the estimate.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
for row, (g, colour) in enumerate(GROUPS.items()):
    x = dfs_events[(dfs_events.group == g) & dfs_events.used]
    m0, length = x.groupby(["m0", "minutes"]).size().idxmax()
    m1, y = m0 + length, 1 - row
    lanes = [  # (start, end, colour, label) in minutes after midnight
        (m0 - 240, m0 - 60, YELLOW, "BL01 reference window" + ("" if row == 0 else " (not used)")),
        (m0 - GAP - REF, m0 - GAP, GRID, "DiD reference"),
        (m1 + GAP, m1 + GAP + REF, GRID, "hour after"),
        (m0, m1, colour, "event"),
    ]
    for a, b, c, label in lanes:
        top = label.startswith("BL01")
        ax.broken_barh([(a / 60, (b - a) / 60)], (y + (0.05 if top else -0.3), 0.25), color=c,
                       alpha=0.35 if (top and row == 1) else 0.9)  # fmt: skip
        ax.text((a + b) / 120, y + (0.17 if top else -0.17), label, ha="center", va="center",
                fontsize=8)  # fmt: skip
    n_typical = x.groupby(["m0", "minutes"]).size().max()
    label = f"{g}\n{m0 // 60:02d}:{m0 % 60:02d}, {length} min ({n_typical} of {len(x)} events)"
    ax.text(10.6, y, label, ha="right", va="center", fontsize=9)
ax.set_xlim(7, 22)
ax.set_ylim(-0.5, 1.5)
ax.set_yticks([])
ax.set_xticks(range(11, 23), [f"{h:02d}:00" for h in range(11, 23)])
ax.set_title("A1 providers learned of the event around 16:30 the day before")
ax.spines[["left"]].set_visible(False);

### Plot 9: the control days, and how unusual the event days were
Left: how many matched days each used event is compared with. Right: how much colder or
warmer each event day was than the nearby non-event days of the same type (before the
2 °C temperature match). Negative = colder than usual for the time of year.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
used = {g: dfs_events[(dfs_events.group == g) & dfs_events.used] for g in GROUPS}
n_matched = [x.n_matched for x in used.values()]
d_temp = [[daily_temp[d] - daily_temp[nearby(d)].mean() for d in x.date] for x in used.values()]
ax1.hist(
    n_matched,
    bins=np.arange(3.5, 22.5),
    color=list(GROUPS.values()),
    label=[f"{g}: median {v.median():.0f}" for g, v in zip(GROUPS, n_matched, strict=True)],
)
ax2.hist(
    d_temp,
    bins=np.arange(-8, 8.5, 1),
    color=list(GROUPS.values()),
    label=[f"{g}: mean {np.mean(v):+.1f} °C" for g, v in zip(GROUPS, d_temp, strict=True)],
)
ax1.set_xlabel("Matched days per event")
ax1.set_ylabel("Events")
ax1.set_title("Size of each event's control group", fontsize=10)
ax2.axvline(0, color=INK2, lw=0.8)
ax2.set_xlabel("Event-day temperature minus nearby non-event days, °C")
ax2.set_title("Were event days unusually cold?", fontsize=10)
for ax in (ax1, ax2):
    ax.legend(fontsize=8)

### What data we have
- **A1 (2022/23): 20 of 21 events.** Only one is dropped (too few matched days). All on
  weekdays, nearly all 60 minutes starting 16:30–18:30, plus two 09:00 morning tests.
  Claims are large: median 184 MW, though the March 2023 tests settled under 1 MW.
- **A2 (Oct 2023+): 66 of 282 events.** Most are lost to two filters: 54 have no settled
  volume published yet (all from June 2025 onwards), and of the rest 113 last longer than
  2 hours (most of them 3–5 hours). Another 34 sit too close to a second event the same day.
  Starts are 16:00–20:00; the median claim is 19 MW, a tenth of A1's. Most used events
  are from 2025.
- **Controls:** both groups have a median of 8 matched days per event.
- **Event-day weather:** relative to nearby non-event days, A1 event days were only
  0.3 °C colder on average and A2 event days 1.0 °C colder. Neither group was called on
  unusually cold days for the time of year, which matters for the BL01 cold-day argument.
- **What differs between the groups going in:** A1 had the in-day adjustment, a day's
  notice and claims about 10× larger, from fewer events. A2 has more events but each is
  small against the event-to-event noise in national data, so its power will be lower
  despite the larger count.